In [1]:
import numpy as np
from itertools import product
from gen.gen_utilities import permutation_symbol as perm
from gen.gen_interpFunction import *
from gen.gen_mesh1D import *
from gen.gen_gaussQuadCalc import *

In [6]:
geom = {'L': 1,   # length of beam, m
        'b': 0.05, # height of beam, m,
        'h': 0.05, # heaight of beam m
        'A1': 0.05, # area of beam in m^2
        'A2': 0.05, # area of beam in m^2
        'A3': 0.0025, # cross-sectional area of beam in m^2
        'I1': 5.20833E-7, # moment of inertia m^4, b*h^3/12
        'I2': 5.20833E-7, # moment of inertia m^4,  inertia about E2xE2
        'J': 1.04166E-6, # polar moment of inertia m^4, pi*(D^4-d^4)/32, inertia about E3xE3, J=I1+I2
        }

material = {'E': 210E9, # Youngs modulus, Pa
                'rho': 7850, # density of steel, kg/m^3
                'G': 80E9, # shear modulus, Pa
                'Ks': 1.2E4 # shear correction coefficient, unitless
                }


NEL = 1
NNPEL = 3
if NNPEL<3:
        NGQP = NNPEL-1 # reduced order integration
else:
        NGQP = NNPEL

appForceGP = np.array([0,0,0], dtype=np.float64)
appMomentGP = np.array([0,0,0], dtype=np.float64)
matModFGP = np.diag([material['G'] * geom['A1'] * material['Ks'], 
                     material['G'] * geom['A2'] * material['Ks'], 
                     material['E'] * geom['A3']
                     ])

matModMGP = np.diag([material['E'] * geom['I1'], 
                     material['E'] * geom['I2'], 
                     material['G'] * geom['J'] 
                     ])

coeffSME = {f'K{i}{j}':np.zeros(shape=[NNPEL,NNPEL],dtype=np.float64) for i,j in product(range(6), repeat=2)} # memory allocation for constants of elemental stiffness matrices
eqns_p_elem = 6 * NNPEL
SME = np.zeros(shape=[eqns_p_elem,eqns_p_elem], dtype=np.float64)

ECON, elemGlobalCoord = mesh1D(1, NEL, NNPEL)
gaussW = gaussLegQuad(NGQP)[1]
print('gaussW=',gaussW)
J = np.dot(elemGlobalCoord,interpLagGLQ(NNPEL, NGQP)[1])
print('J=',J)


gaussW= [0.55555556 0.88888889 0.55555556]
J= [[0.5 0.5 0.5]]


In [8]:
phi = np.array([[0,0,0],[0,0,0.5],[0,0,1]],dtype=np.float64)
# phi = np.array([[0,0,0],[0,0,1]],dtype=np.float64)
for GP in range(NGQP):
    effecWt = J[0,GP] * gaussW[GP]
    print('effecWt=',effecWt)
    S0 = interpLagGLQ(NNPEL, NGQP)[0][:,GP]
    S1 = interpLagGLQ(NNPEL, NGQP)[1][:,GP] * (1/J[0,GP])
    
    S00 = np.outer(S0,S0)
    S11 = np.outer(S1,S1)
    S10 = np.outer(S1,S0)
    print('S00=',S00)            
    print('S10=',S10)            
    print('S11=',S11)            
    omegaGP = np.array([[[0,0,0]],[[0,0,0]]],dtype=np.float64)
    
    dphio = np.dot(S1,phi)
    print('dphio=',dphio)
    rotation = np.array([[1,0,0],[0,1,0],[0,0,1]], dtype=np.float64)
    gammaGP = dphio - (rotation @ np.array([0,0,1]))
    print('gammaGP=',gammaGP)
    forceGP = matModFGP @ gammaGP
    momentGP = matModMGP @ omegaGP[0,0,:]

    
    K11a3b3 = 0.0

    for a in range(3): # a=alpha, b=beta as notes in reference material
        for b in range(3):
            coeffSME[f'K{a}{b}'][:,:] += matModFGP[a,b] * S11 * effecWt
            print(f'coeffSME[K{a}{b}]=',coeffSME[f'K{a}{b}'])
            print(f'coeffSME[K{a}{b}][{GP}]=',matModFGP[a,b] * S11 * effecWt)
            print(f'matModFGP[{a},{b}]=',matModFGP[a,b])
            print('S11=',S11)
            print('effWt=',effecWt)
            # print(f'coeffSME[K{a}{b}]=',coeffSME[f'K{a}{b}'])
            K11a3b3 = matModMGP[a,b]
            K10ab3 = 0.0
            K01a3b = 0.0
            K00a3b3_0 = 0.0
            K00a3b3_1 = 0.0
            K10a3b3 = 0.0
            for k in range(3):
                for l in range(3):
                    K10ab3 += perm()[k,b,l] * dphio[k]*matModFGP[a,l] - perm()[k,b,a]*forceGP[k]
                    K01a3b += perm()[k,a,l] * dphio[k]*matModFGP[b,l] + perm()[k,b,a]*forceGP[k]
                    K10a3b3 += - perm()[k,b,a]*momentGP[k]
                    for m in range(3):
                        K00a3b3_1 += perm()[a,k,l]*perm()[m,b,l]*dphio[k]*forceGP[m]
                        for n in range(3):
                            K00a3b3_0 += perm()[k,a,l]*perm()[m,b,n]*dphio[k]*dphio[m]*matModFGP[l,n]
            # print('a=',a,'b=',b,'K01a3b=',K01a3b,'K10ab3=',K10ab3)
            coeffSME[f'K{a}{b+3}'][:,:] += K10ab3 * S10 * effecWt
            # print(f'coeffSME[K{a}{b+3}]=',coeffSME[f'K{a}{b+3}'])
            coeffSME[f'K{a+3}{b}'][:,:] += K01a3b * S10.T * effecWt
            # print(f'coeffSME[K{a+3}{b}]=',coeffSME[f'K{a+3}{b}'])
            coeffSME[f'K{a+3}{b+3}'][:,:] += (K11a3b3 * S11 + K10a3b3*S10 + (K00a3b3_0+K00a3b3_1)*S00) * effecWt
            # print(f'coeffSME[K{a+3}{b+3}]=',coeffSME[f'K{a+3}{b+3}'])

print('coeffSME=',coeffSME)   
print('det coeffSME 00=',np.linalg.det(coeffSME['K00']))
print('det coeffSME 11=',np.linalg.det(coeffSME['K11']))
print('det coeffSME 22=',np.linalg.det(coeffSME['K22']))
print('det coeffSME 33=',np.linalg.det(coeffSME['K33']))
print('det coeffSME 44=',np.linalg.det(coeffSME['K44']))
print('det coeffSME 55=',np.linalg.det(coeffSME['K55']))
print('det coeffSME 04=',np.linalg.det(coeffSME['K04']))
print('det coeffSME 13=',np.linalg.det(coeffSME['K13']))


effecWt= 0.2777777777777779
S00= [[ 0.472379    0.27491933 -0.06      ]
 [ 0.27491933  0.16       -0.03491933]
 [-0.06       -0.03491933  0.007621  ]]
S10= [[-1.75205634 -1.01967734  0.22254033]
 [ 2.129516    1.23935467 -0.270484  ]
 [-0.37745967 -0.21967734  0.04794366]]
S11= [[ 6.49838668 -7.89838668  1.4       ]
 [-7.89838668  9.6        -1.70161332]
 [ 1.4        -1.70161332  0.30161332]]
dphio= [0. 0. 1.]
gammaGP= [0. 0. 0.]
coeffSME[K00]= [[ 8.66451557e+13 -1.05311822e+14  1.86666667e+13]
 [-1.05311822e+14  1.28000000e+14 -2.26881776e+13]
 [ 1.86666667e+13 -2.26881776e+13  4.02151097e+12]]
coeffSME[K00][0]= [[ 8.66451557e+13 -1.05311822e+14  1.86666667e+13]
 [-1.05311822e+14  1.28000000e+14 -2.26881776e+13]
 [ 1.86666667e+13 -2.26881776e+13  4.02151097e+12]]
matModFGP[0,0]= 48000000000000.0
S11= [[ 6.49838668 -7.89838668  1.4       ]
 [-7.89838668  9.6        -1.70161332]
 [ 1.4        -1.70161332  0.30161332]]
effWt= 0.2777777777777779
coeffSME[K01]= [[0. 0. 0.]
 [0. 0. 0.]
 [0

In [6]:
for j in range(6):
    for k in range(6):
        SME[j:eqns_p_elem:6, k:eqns_p_elem:6] = coeffSME[f'K{j}{k}']

print('detSME=',np.linalg.det(SME))

detSME= -2.2146972716956075e+95


In [97]:
coeffCVE = {f'F{i}':np.zeros(shape=[2],dtype=np.float64) for i in range(6)} # memory allocation for constants of elemental stiffness matrices
F0a = 0.0
F1a = 0.0
F1a3 = 0.0
for a in range(3):
    F0a = appForceGP[a]
    F1a = -forceGP[a]
    coeffCVE[f'F{a}'][:] = (F0a*S0 + F1a*S1) * effecWt
    print(f'coeffCVE[F{a}]=',coeffCVE[f'F{a}'])
    F1a3 = -momentGP[a]
    F0a3 = 0.0
    for k in range(3):
        for l in range(3):
            F0a3 += perm()[a,l,k] * forceGP[k] * dphio[l] + appMomentGP[a]
    coeffCVE[f'F{a+3}'][:] = (F0a3*S0 + F1a3*S1) * effecWt
    print(f'coeffCVE[F{a+3}]=',coeffCVE[f'F{a+3}'])
    




coeffCVE[F0]= [0. 0.]
coeffCVE[F3]= [0. 0.]
coeffCVE[F1]= [0. 0.]
coeffCVE[F4]= [0. 0.]
coeffCVE[F2]= [-4999.99999801  4999.99999801]
coeffCVE[F5]= [0. 0.]
